In [1]:
import pandas as pd
import os
import pickle
from sklearn.metrics import classification_report

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [2]:
# Load Data
df = pd.read_csv("../data/student_dropout_dataset_v3.csv")

In [3]:
df.drop("Student_ID", axis=1, inplace=True)

# Handle Missing Values
df.fillna(df.mean(numeric_only=True), inplace=True)


In [4]:
# Encode Categorical
encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le


In [5]:

# Split
X = df.drop("Dropout", axis=1)
y = df["Dropout"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [6]:
# XGBoost Model
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [7]:

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))



Accuracy: 0.801


In [8]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=2,   # 🔥 IMPORTANT (imbalance fix)
    random_state=42
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.84      0.80      1529
           1       0.22      0.15      0.18       471

    accuracy                           0.68      2000
   macro avg       0.49      0.50      0.49      2000
weighted avg       0.64      0.68      0.65      2000



In [11]:
from collections import Counter

counter = Counter(y)
print(counter)

scale = counter[0] / counter[1]
print("scale_pos_weight:", scale)

Counter({0: 7646, 1: 2354})
scale_pos_weight: 3.248088360237893


In [12]:
model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale,
    random_state=42
)

In [13]:
# 1. train model
model.fit(X_train, y_train)

# 2. then probability
y_prob = model.predict_proba(X_test)[:, 1]

# 3. threshold
y_pred = (y_prob > 0.35).astype(int)

In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.77      0.82      1529
           1       0.48      0.69      0.57       471

    accuracy                           0.75      2000
   macro avg       0.68      0.73      0.69      2000
weighted avg       0.79      0.75      0.76      2000



In [15]:
for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    from sklearn.metrics import classification_report
    print(classification_report(y_test, y_pred))


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1529
           1       0.46      0.72      0.56       471

    accuracy                           0.73      2000
   macro avg       0.68      0.73      0.68      2000
weighted avg       0.79      0.73      0.75      2000


Threshold: 0.35
              precision    recall  f1-score   support

           0       0.89      0.77      0.82      1529
           1       0.48      0.69      0.57       471

    accuracy                           0.75      2000
   macro avg       0.68      0.73      0.69      2000
weighted avg       0.79      0.75      0.76      2000


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1529
           1       0.50      0.65      0.56       471

    accuracy                           0.76      2000
   macro avg       0.69      0.72      0.70      2000
weighted avg       0.79  

In [16]:
y_prob = model.predict_proba(X_test)[:, 1]

# FINAL threshold
y_pred = (y_prob > 0.40).astype(int)

In [17]:
# Save Everything
os.makedirs("../model", exist_ok=True)

with open("../model/model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("../model/encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

print("Model + Encoders saved ✅")

Model + Encoders saved ✅
